# 04 · Evaluation: score files and leave-one-scenario-out (T6)

Owner: Ayo (Y16). The one results machine for T5, T6, T8 and F4–F6 (TEAM_PLAN 2.5).

Order: make sure every final score file exists (running any missing `final` command, so nothing depends on received files) → `evaluate_file()` on each (M3 check) → LOSO on v1 with the tuned defaults and each model's set-up (`ms.FINAL_SETUP`) → `loso.csv` (T6) and the PR-AUC pivot.

Run with Restart & Run All. LOSO takes about 20 minutes (6 fits; Random Forest with SMOTE is the slow one). It reruns only if `loso.csv` is missing or `RERUN_LOSO = True`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src import models_supervised as ms
from src import models_unsupervised as mu
from src.config import PROCESSED_DIR, SCORES_DIR, TABLES_DIR
from src.evaluate import evaluate_file

FEATURES = PROCESSED_DIR / "features_v1.parquet"
RERUN_LOSO = False
UNSUPERVISED = ["iforest", "autoencoder", "iforest_contam", "autoencoder_contam", "ocsvm"]
SUPERVISED = list(ms.MODELS)
df = pd.read_parquet(FEATURES)
print(f"{FEATURES.name}: {len(df):,} user-days, {df['label'].sum():,} malicious")

features_v1.parquet: 330,268 user-days, 984 malicious


## 1 · Final score files
Runs `python -m src.models_unsupervised final` / `python -m src.models_supervised final` only for a family with a missing file.

In [2]:
def missing(names):
    return [n for n in names if not (SCORES_DIR / f"{n}.parquet").exists()]

if missing(UNSUPERVISED):
    print("running the unsupervised final for", missing(UNSUPERVISED))
    mu.final(df=df)
if missing(SUPERVISED):
    print("running the supervised final for", missing(SUPERVISED))
    ms.final(df=df)
assert not missing(UNSUPERVISED + SUPERVISED)
print("all", len(UNSUPERVISED + SUPERVISED), "final score files present")

all 7 final score files present


## 2 · `evaluate_file()` on every score file (M3)
Test split only. Shown to check that each file evaluates; the comparison itself is Y17 (T5).

In [3]:
rows = []
for name in UNSUPERVISED + SUPERVISED:
    result = evaluate_file(SCORES_DIR / f"{name}.parquet")
    rows.append({"model": name, "test_pr_auc": result["ranking"]["pr_auc"],
                 "test_roc_auc": result["ranking"]["roc_auc"]})
pd.DataFrame(rows).round(4)

,model,test_pr_auc,test_roc_auc
0,iforest,0.0122,0.8792
1,autoencoder,0.0930,0.9160
2,iforest_contam,0.0101,0.8606
3,autoencoder_contam,0.0413,0.9018
4,ocsvm,0.0574,0.7412
5,random_forest,0.9658,0.9999
6,xgboost,0.9877,0.9997


## 3 · LOSO (T6)
Per held-out scenario: RF and XGBoost trained without that scenario's accounts (tuned defaults, `FINAL_SETUP`), and the Isolation Forest and autoencoder final score files evaluated on the same test rows. Writes `results/tables/loso.csv`, `loso_pr_auc.csv` and `results/scores/<model>_loso_s<1|2|3>.parquet`.

In [4]:
if RERUN_LOSO or not (TABLES_DIR / ms.LOSO_TABLE).exists():
    loso = ms.loso(df=df)
else:
    loso = pd.read_csv(TABLES_DIR / ms.LOSO_TABLE)
    print(f"read {ms.LOSO_TABLE}; set RERUN_LOSO = True to recompute")
for model in SUPERVISED:
    for s in (1, 2, 3):
        assert (SCORES_DIR / f"{model}_loso_s{s}.parquet").exists()
loso.round(4)

read loso.csv; set RERUN_LOSO = True to recompute


,held_out,model,kind,n_train,held_out_accounts,n_rows,n_malicious,pr_auc,roc_auc,recall_top1pct
0,1,random_forest,supervised (LOSO),203199.0,30.0,64846,9,0.1678,0.9925,0.7778
1,1,xgboost,supervised (LOSO),203199.0,30.0,64846,9,0.5011,0.9267,0.7778
2,1,iforest,unsupervised,NaN,NaN,64846,9,0.0043,0.9751,0.3333
3,1,autoencoder,unsupervised,NaN,NaN,64846,9,0.9234,1.0000,1.0000
4,2,random_forest,supervised (LOSO),203144.0,30.0,64994,157,0.0322,0.8154,0.2229
5,2,xgboost,supervised (LOSO),203144.0,30.0,64994,157,0.0030,0.5596,0.0000
6,2,iforest,unsupervised,NaN,NaN,64994,157,0.0101,0.8770,0.0318
7,2,autoencoder,unsupervised,NaN,NaN,64994,157,0.0240,0.9139,0.1274
8,3,random_forest,supervised (LOSO),206909.0,12.0,64841,4,0.2519,0.7191,0.5000
9,3,xgboost,supervised (LOSO),206909.0,12.0,64841,4,0.2507,0.6641,0.2500


In [5]:
pivot = ms.loso_pivot(loso)        # PR-AUC per held-out scenario, saved as loso_pr_auc.csv
pivot.round(4)

,kind,s1,s2,s3
model,,,,
random_forest,supervised (LOSO),0.1678,0.0322,0.2519
xgboost,supervised (LOSO),0.5011,0.0030,0.2507
iforest,unsupervised,0.0043,0.0101,0.0035
autoencoder,unsupervised,0.9234,0.0240,0.3383


In [6]:
# test rows per held-out scenario (the same for every model); scenario 3 has very few malicious days
loso.groupby("held_out")[["n_rows", "n_malicious"]].first()

,n_rows,n_malicious
held_out,,
1,64846,9
2,64994,157
3,64841,4


## 4 · Reading [H]

*Drafted by Claude at Ayo's request (Nov 9); numbers updated Nov 10 for the tuned Isolation Forest and autoencoder defaults (#50). Ayo to check it against the tables above. Full-training numbers: per-scenario test PR-AUC from `evaluate_file()` on `random_forest.parquet` / `xgboost.parquet`, the same test rows as LOSO.*

Yes, supervised PR-AUC drops sharply on the unseen scenario. With scenario 1 held out, Random Forest falls from 0.87 to 0.17 and XGBoost from 0.90 to 0.50. On scenario 2 both collapse: RF goes from 0.97 to 0.03, and XGBoost from 0.99 to 0.003, with ROC-AUC 0.56, close to chance. On scenario 3, XGBoost falls from 0.80 to 0.25 and RF from 0.32 to 0.25.

The unsupervised models never see labels, so their scores are identical with or without LOSO, but they are not uniformly good. The autoencoder keeps 0.92 on scenario 1 and 0.34 on scenario 3, ahead of both supervised models trained without those scenarios. Isolation Forest stays at about 0.01 or below everywhere.

No model detects an unseen scenario 2 well. The best is RF at 0.03, catching 22% of its 157 days in the top 1%.

These numbers rest on very few positives: 9 malicious test days for scenario 1 and only 4 for scenario 3 (one real insider with 2 days, plus 2 days on the hijacked `FBA0348` account). A single day moves PR-AUC a lot, so the scenario-3 results are indicative only and belong in Limitations.